# Gemma 4 Developer Agent Submission

Single-agent Gemma 4 SWE agent for the *Gemma 4 Developer Agent* competition.

**What this notebook does**
1. Writes the agent (`agent.yaml`, prompt, sampling and budget configs) to `/kaggle/working/submission` and packages `/kaggle/working/submission.zip`.
2. *Interactive runs only:* installs the official harness, validates and compiles the submission, starts vLLM and runs the agent on a few public tasks to estimate the resolution rate.

When Kaggle re-runs this notebook for scoring, step 2 is skipped automatically, so only the submission files are produced.

**Setup:** accelerator **GPU L4 x4**, internet **off**. Inputs: the competition data, the `metric/gemma-4-developer-agent-wheelhouse` dataset and the `google/gemma-4` model matching `model:` in `agent.yaml`.

**Submit:** after a run, use *Submit to Competition* on the notebook (1 submission per day).

In [ ]:
import json
import os
from pathlib import Path

# Run the agent on a few public tasks to estimate the resolution rate (interactive runs only)
RUN_DEV_EVAL = True
# Before the agent runs, grade each dev task's reference solution to confirm local grading works (CPU only)
RUN_HARNESS_CHECK = False  # dev tasks are already gold-validated (dev/validated_tasks.txt)
# Only grade the reference solutions of ALL public tasks (CPU only, no model, no agent)
GOLD_CHECK_ONLY = False
# Record the working-tree diff after every tool call (for the anytime curve; dev runs only)
RECORD_ANYTIME = True
# Tasks evaluated in parallel during dev runs (the scorer runs them one at a time)
DEV_CONCURRENCY = 1
# Tasks per repo for the dev run; set DEV_TASK_IDS to a list of instance_ids to pick them yourself
DEV_TASKS_PER_REPO = {'fastapi/fastapi': 2, 'Textualize/rich': 2, 'psf/requests': 1, 'encode/httpx': 1}
DEV_TASK_IDS = ['fastapi_11194', 'fastapi_14372', 'fastapi_14492', 'fastapi_15280', 'fastapi_9425', 'rich_3043', 'rich_3278', 'rich_3506', 'rich_3772', 'rich_3934']

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
WORKING_DIR = Path('/kaggle/working')
AGENT_DIR = WORKING_DIR / 'submission'


def is_scoring_rerun() -> bool:
    """True when Kaggle re-runs the notebook for scoring (hidden tasks carry no test_patch)."""
    if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
        return True
    try:
        with open(DATA_DIR / 'tasks.jsonl', encoding='utf-8') as f:
            return 'test_patch' not in json.loads(f.readline())
    except (OSError, ValueError):
        return False


IS_RERUN = is_scoring_rerun()
DEV = RUN_DEV_EVAL and not IS_RERUN
print(f'Scoring re-run: {IS_RERUN} | dev evaluation: {DEV}')

for sub in ('configs', 'prompts', 'skills/swe-tools', 'skills/swe-tools/scripts'):
    (AGENT_DIR / sub).mkdir(parents=True, exist_ok=True)

## 1. Submission files
Edit these cells to change the agent. `model:` in `agent.yaml` must name one of the four allowed quantized models.

In [ ]:
%%writefile /kaggle/working/submission/agent.yaml
name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - edit_file
  - write_file
  - get_status
  - submit_patch
skills:
  - skills/swe-tools
generate_content_config: !include configs/sampling.yaml


In [ ]:
%%writefile /kaggle/working/submission/eval_config.yaml
# Wall-clock is the binding limit: ~120 tasks x (4.5 min + ~0.5 min setup) = 10 h worst case, under the 12 h cap.
# The budget numbers are repeated in prompts/system.md -- keep them in sync.
evaluation:
  timeout_seconds: 90
  max_tool_calls: 30
  max_time_minutes: 4.5
  max_turns: 60


In [ ]:
%%writefile /kaggle/working/submission/configs/sampling.yaml
temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 2048
  include_thoughts: true


In [ ]:
%%writefile /kaggle/working/submission/prompts/system.md
You are an expert software engineer fixing an issue in the Python repository at /workspace. Hidden tests, written from the issue text, grade your change: the exact names, signatures, messages, exception types and return values in the issue are the specification.

# Budget: 4.5 minutes (each step takes about 10 seconds), at most 30 tool calls
When time runs out, the working tree is graded as it is: an edit that exists counts even without `submit_patch`; an unedited tree scores zero. Time, not the call count, is your real limit, so think briefly and act.

# Step 1 (no tool call): the contract
Write a short checklist from the issue: every required file, function, class, method, parameter and default (exact spelling); exact error messages and exception types; return values; edge cases. Quoted text is literal: `"None"` in quotes is the string "None".

# Step 2 (call 1): locate with the skill
Call `run_skill_script` with skill_name `swe-tools`, file_path `scripts/start.py` and args = one list holding a single string with the identifiers, quoted error text and key words from the issue. It prints the most likely functions (`path:line name`) and files.

# Step 3 (calls 2-6): read only what you need
- Read a candidate: `python3 .git/swe/view.py <path> <Name or Class.method or START-END>` (at most 80 lines). Read two candidates in one call by joining commands with `;`.
- If the candidates look wrong: `python3 .git/swe/locate.py "<other words>"` or `git grep -n "exact text" -- '*.py' | head -20`.
- Never print whole files or long outputs; pipe through `| head -40`.

# Step 4 (by call 8 at the latest): edit
- Implement every contract item at the place where the wrong behaviour is produced. For a feature, copy the nearest sibling pattern, including imports, `__all__`, registries and every parallel implementation (sync/async, overloads, both app and router variants).
- `edit_file` needs `old_string` copied exactly from text you printed in this task, short and unique. If it fails, print those lines again with view.py and retry with a shorter snippet; never repeat an identical failing call. Use `write_file` only for a new file the issue requires.
- A partial edit always beats no edit. Make your best-guess edit by call 8 even if unsure, then refine.

# Step 5: check and submit
- `python3 .git/swe/check.py` (compiles changed files, flags discarded or stray files, names related tests). Fix any PROBLEMS it lists.
- Optionally run one related test: `timeout 80 python3 -m pytest <test file> -x -q 2>&1 | tail -15`. Failures unrelated to your change are pre-existing; ignore them.
- Call `submit_patch`, confirm `patch_size > 0`, reply with one sentence.

# Rules
- Changes to tests, `conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini` are discarded before grading. An issue about tests, CI or versions still needs a source change.
- Scratch files go in /tmp only. Never pip install; there is no network.
- If a message starts with "For context:", it summarizes YOUR OWN earlier steps in this task. Continue from it; do not redo those steps.
- Never conclude that nothing needs to change.


In [ ]:
%%writefile /kaggle/working/submission/skills/swe-tools/SKILL.md
---
name: swe-tools
description: Fast repository navigation for fixing an issue. start.py installs helpers into .git/swe and prints the functions most likely to need a change.
---
# swe-tools

Run once, as your first action:
run_skill_script(skill_name="swe-tools", file_path="scripts/start.py", args=["<identifiers, quoted error text and key words from the issue>"])

It prints the ten most likely functions or classes (path:line and name) and the most likely files, and installs three helpers that you then call with run_command:
- `python3 .git/swe/locate.py "<different keywords>"` searches again with other words.
- `python3 .git/swe/view.py <path> <Name|Class.method|START-END>` prints one definition or line range (at most 80 lines) with line numbers.
- `python3 .git/swe/check.py` lists changed files, compiles them, flags discarded test/config edits and stray files, and names related test files.

The helpers live inside .git, so they are never part of your patch.


In [ ]:
%%writefile /kaggle/working/submission/skills/swe-tools/scripts/check.py
"""Pre-submit check: what changed, does it compile, anything that will be discarded, which tests to run.

Usage:  python3 .git/swe/check.py
"""
import os
import py_compile
import re
import subprocess

PROTECTED = re.compile(r'(^|/)(tests?|testing)/.*\.py$|(^|/)test_[^/]*\.py$|_test\.py$|(^|/)conftest\.py$|'
                       r'(^|/)(pytest\.ini|pyproject\.toml|setup\.cfg|tox\.ini|\.pytest\.ini)$')


def repo_root():
    for cand in (os.environ.get('SWE_REPO'), os.environ.get('PWD'), '/workspace', os.getcwd()):
        if cand and os.path.isdir(os.path.join(cand, '.git')):
            return cand
    return os.getcwd()


def git(root, *args):
    return subprocess.run(['git', *args], cwd=root, capture_output=True, text=True).stdout


def main():
    root = repo_root()
    status = [l for l in git(root, 'status', '--porcelain').splitlines() if l.strip()]
    if not status:
        print('NO CHANGES. Nothing will be graded: edit a source file now.')
        return
    changed, problems = [], []
    for line in status:
        code, path = line[:2], line[3:].strip().strip('"')
        if path.endswith('/'):
            problems.append(f'untracked directory {path} will be part of the patch; remove it if it is scratch')
            continue
        changed.append(path)
        if PROTECTED.search(path):
            problems.append(f'{path}: test/config file, discarded before grading (revert: git checkout -- {path})')
        elif code == '??' and not path.endswith('.py'):
            problems.append(f'{path}: untracked non-Python file, will be part of the patch; delete if scratch')
    for path in changed:
        if path.endswith('.py') and os.path.exists(os.path.join(root, path)):
            try:
                py_compile.compile(os.path.join(root, path), doraise=True)
            except py_compile.PyCompileError as e:
                problems.append(f'{path}: SYNTAX ERROR {str(e.msg).strip()[:300]}')
    print('Changed: ' + ', '.join(changed))
    print(git(root, 'diff', '--stat').strip()[-800:])
    src = [p for p in changed if p.endswith('.py') and not PROTECTED.search(p)]
    if not src:
        problems.append('no source (.py outside tests) file changed: the graded patch would be empty')
    tests = []
    for p in src:
        stem = os.path.basename(p)[:-3].lstrip('_')
        for t in git(root, 'ls-files').splitlines():
            base = os.path.basename(t)
            if base.startswith('test_') and stem and stem in base and t.endswith('.py'):
                tests.append(t)
    if tests:
        print('Related tests: ' + ' '.join(sorted(set(tests))[:4]))
        print('Run one: timeout 120 python3 -m pytest <file> -x -q 2>&1 | tail -15')
    print('PROBLEMS:\n- ' + '\n- '.join(problems) if problems else 'OK: source changed and compiles.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/submission/skills/swe-tools/scripts/locate.py
"""Rank the functions/classes of the repository most likely to need a change for an issue.

Usage:  python3 .git/swe/locate.py "<issue keywords, identifiers, error message>"
        python3 .git/swe/locate.py --file issue.txt          (offline evaluation)
Prints the top candidates as `path:line  qualified.name` with their signature, plus the top files.
Scoring: BM25 over (path, name, body) of every AST chunk, fused (RRF) with exact identifier and
quoted-string matches. Standard library only.
"""
import ast
import collections
import math
import os
import re
import sys

SKIP_DIRS = {'.git', '__pycache__', 'node_modules', '.venv', 'venv', 'env', 'build', 'dist', '.tox',
             'site-packages', '.mypy_cache', '.pytest_cache', '.eggs'}
TEST_RE = re.compile(r'(^|/)(tests?|testing)/|(^|/)test_[^/]*\.py$|_test\.py$|(^|/)conftest\.py$')
MAX_FILE_BYTES = 300_000
WORD = re.compile(r'[A-Za-z_][A-Za-z0-9_]*')
PART = re.compile(r'[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|\d+')
STOP = set('the and for that this with from into when then than have has not are was were will would '
           'should could can may use used using get set def class self none true false return import '
           'issue fix bug error test tests example add new support value values'.split())


def repo_root():
    for cand in (os.environ.get('SWE_REPO'), os.environ.get('PWD'), '/workspace', os.getcwd()):
        if cand and os.path.isdir(os.path.join(cand, '.git')):
            return cand
    return os.getcwd()


def tokens(text):
    out = []
    for w in WORD.findall(text):
        lw = w.lower()
        if len(lw) > 2 and lw not in STOP:
            out.append(lw)
        parts = [p.lower() for p in PART.findall(w.replace('_', ' ')) if len(p) > 2]
        if len(parts) > 1:
            out.extend(p for p in parts if p not in STOP)
    return out


def chunks_of(path, rel):
    try:
        src = open(path, encoding='utf-8', errors='replace').read()
        tree = ast.parse(src)
    except (SyntaxError, ValueError, OSError):
        return []
    lines = src.splitlines()
    module = rel[:-3].replace('/', '.')
    out = [{'rel': rel, 'line': 1, 'name': module, 'kind': 'module',
            'sig': (lines[0] if lines else ''), 'text': ' '.join(lines[:40])}]

    def visit(node, prefix):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                qual = f'{prefix}{child.name}'
                start, end = child.lineno, getattr(child, 'end_lineno', child.lineno)
                if isinstance(child, ast.ClassDef):
                    body_end = min(end, start + 25)  # class header + docstring; methods are own chunks
                else:
                    body_end = end
                text = '\n'.join(lines[start - 1:body_end])
                out.append({'rel': rel, 'line': start, 'name': qual, 'kind': type(child).__name__,
                            'sig': lines[start - 1].strip() if start - 1 < len(lines) else qual,
                            'text': text})
                if isinstance(child, ast.ClassDef):
                    visit(child, qual + '.')
    visit(tree, '')
    return out


def collect(root):
    chunks = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith('.')]
        for fn in filenames:
            if not fn.endswith('.py'):
                continue
            path = os.path.join(dirpath, fn)
            rel = os.path.relpath(path, root).replace(os.sep, '/')
            if TEST_RE.search(rel):
                continue
            try:
                if os.path.getsize(path) > MAX_FILE_BYTES:
                    continue
            except OSError:
                continue
            chunks.extend(chunks_of(path, rel))
    return chunks


def bm25_rank(query_toks, docs, k1=1.2, b=0.75):
    tfs = [collections.Counter(d) for d in docs]
    lens = [len(d) for d in docs]
    avg = (sum(lens) / len(lens)) if lens else 1.0
    df = collections.Counter(t for tf in tfs for t in tf)
    n = len(docs)
    q = collections.Counter(query_toks)
    scores = []
    for tf, ln in zip(tfs, lens):
        s = 0.0
        for t, qn in q.items():
            f = tf.get(t)
            if not f:
                continue
            idf = math.log(1 + (n - df[t] + .5) / (df[t] + .5))
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * ln / avg)) * (1 + 0.2 * (qn - 1))
        scores.append(s)
    return scores


def main():
    args = sys.argv[1:]
    if args[:1] == ['--file']:
        query = open(args[1], encoding='utf-8', errors='replace').read()
        args = args[2:]
    else:
        query = ' '.join(a for a in args if a != '--')
    top_n = 10
    root = repo_root()
    chunks = collect(root)
    if not chunks or not query.strip():
        print('No Python sources found or empty query. Usage: locate.py "<keywords>"')
        return
    q_toks = tokens(query)
    idents = {w for w in WORD.findall(query) if len(w) > 3}
    quoted = [s.strip() for s in re.findall(r'[\'"`]([^\'"`\n]{6,80})[\'"`]', query)]
    docs = [tokens(c['rel'].replace('/', ' ')) + tokens(c['name']) * 3 + tokens(c['text']) for c in chunks]
    bm = bm25_rank(q_toks, docs)
    exact = []
    for c in chunks:
        last = c['name'].rsplit('.', 1)[-1]
        s = 0.0
        if last in idents:
            s += 3.0
        if any(part in idents for part in c['name'].split('.')[:-1]):
            s += 0.3
        stem = c['rel'].rsplit('/', 1)[-1][:-3]
        if stem in idents or c['rel'] in query:
            s += 1.0
        s += sum(1.5 for qs in quoted if qs in c['text'])
        exact.append(s)
    rrf = collections.defaultdict(float)
    for scores in (bm, exact):
        order = sorted(range(len(chunks)), key=lambda i: -scores[i])
        for rank, i in enumerate(order[:300]):
            if scores[i] > 0:
                rrf[i] += 1.0 / (30 + rank)
    for i, s in enumerate(exact):  # strong exact evidence should lead
        rrf[i] += 0.02 * s
    ranked = sorted(rrf, key=rrf.get, reverse=True)
    funcs, per_file = [], collections.Counter()
    for i in ranked:  # at most 3 candidates per file keeps the shortlist diverse
        c = chunks[i]
        if c['kind'] != 'module' and per_file[c['rel']] < 3:
            funcs.append(i)
            per_file[c['rel']] += 1
        if len(funcs) >= top_n:
            break
    print(f'Top {len(funcs)} candidate locations (most likely first):')
    for i in funcs:
        c = chunks[i]
        print(f"  {c['rel']}:{c['line']}  {c['name']}\n      {c['sig'][:110]}")
    files = collections.OrderedDict()
    for i in ranked:
        files.setdefault(chunks[i]['rel'], 0)
        if len(files) >= 6:
            break
    print('Top files: ' + ', '.join(files))
    print('Next: python3 .git/swe/view.py <path> <name or start-end> to read one candidate.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/submission/skills/swe-tools/scripts/start.py
"""Install the helper scripts into <repo>/.git/swe/ (never part of the patch) and run the localizer.

Run once, as the first action:  run_skill_script(skill_name="swe-tools", file_path="scripts/start.py",
                                                 args=["<keywords from the issue>"])
Afterwards use run_command:  python3 .git/swe/locate.py "...", .git/swe/view.py, .git/swe/check.py
"""
import os
import runpy
import shutil
import sys

HERE = os.path.dirname(os.path.abspath(__file__))


def repo_root():
    for cand in (os.environ.get('SWE_REPO'), os.environ.get('PWD'), '/workspace'):
        if cand and os.path.isdir(os.path.join(cand, '.git')):
            return cand
    return None


root = repo_root()
if root is None:
    print('Repository not found (expected /workspace).')
    sys.exit(0)
dest = os.path.join(root, '.git', 'swe')
os.makedirs(dest, exist_ok=True)
for name in ('locate.py', 'view.py', 'check.py'):
    shutil.copyfile(os.path.join(HERE, name), os.path.join(dest, name))
print('Installed: python3 .git/swe/locate.py "<keywords>" | .git/swe/view.py <path> <Name|A-B> | .git/swe/check.py')
os.environ['SWE_REPO'] = root
sys.argv = [os.path.join(dest, 'locate.py')] + [a for a in sys.argv[1:] if a != '--']
runpy.run_path(os.path.join(dest, 'locate.py'), run_name='__main__')


In [ ]:
%%writefile /kaggle/working/submission/skills/swe-tools/scripts/view.py
"""Print one function/class (or a line range) of a file with line numbers.

Usage:  python3 .git/swe/view.py <path> <Name|Class.method|START-END> [max_lines]
At most 80 lines are printed; the header says which lines were shown and how many exist.
"""
import ast
import os
import sys

MAX_LINES = 80


def repo_root():
    for cand in (os.environ.get('SWE_REPO'), os.environ.get('PWD'), '/workspace', os.getcwd()):
        if cand and os.path.isdir(os.path.join(cand, '.git')):
            return cand
    return os.getcwd()


def find_symbol(tree, target):
    parts = target.split('.')
    best = None

    def visit(node, prefix):
        nonlocal best
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                qual = prefix + [child.name]
                if qual[-len(parts):] == parts and best is None:
                    best = child
                visit(child, qual)
    visit(tree, [])
    return best


def main():
    if len(sys.argv) < 2:
        print(__doc__)
        return
    path = sys.argv[1].lstrip('/')
    if path.startswith('workspace/'):
        path = path[len('workspace/'):]
    full = os.path.join(repo_root(), path)
    if not os.path.isfile(full):
        print(f'No such file: {path}. Find it with: git ls-files | grep <name>')
        return
    lines = open(full, encoding='utf-8', errors='replace').read().splitlines()
    limit = int(sys.argv[3]) if len(sys.argv) > 3 and sys.argv[3].isdigit() else MAX_LINES
    target = sys.argv[2] if len(sys.argv) > 2 else '1-80'
    start, end = 1, min(len(lines), limit)
    if '-' in target and target.replace('-', '').isdigit():
        a, b = target.split('-', 1)
        start, end = max(1, int(a)), min(len(lines), int(b))
    elif target.isdigit():
        start = max(1, int(target) - 10)
        end = min(len(lines), start + 40)
    else:
        try:
            node = find_symbol(ast.parse('\n'.join(lines)), target)
        except SyntaxError:
            node = None
        if node is None:
            hits = [i + 1 for i, l in enumerate(lines) if target.split('.')[-1] in l][:8]
            print(f'{target} not found as a definition in {path}. Lines mentioning it: {hits}')
            return
        start = node.lineno - len(getattr(node, 'decorator_list', []) or [])
        end = getattr(node, 'end_lineno', node.lineno)
    shown_end = min(end, start + limit - 1)
    print(f'{path} lines {start}-{shown_end} of {len(lines)}' +
          (f' (truncated; definition ends at {end}, view {shown_end + 1}-{end} next)' if shown_end < end else ''))
    for i in range(start, shown_end + 1):
        print(f'{i:5d}  {lines[i - 1]}')


if __name__ == '__main__':
    main()


## 2. Package `submission.zip`

In [ ]:
import zipfile

ALLOWED_EXTENSIONS = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}

# Files only (no directory entries), agent.yaml at the archive root
files = sorted(p for p in AGENT_DIR.rglob('*') if p.is_file())
for p in files:
    assert not p.is_symlink(), f'Symlinks are not permitted: {p}'
    assert p.suffix.lower() in ALLOWED_EXTENSIONS, f'Disallowed file extension: {p}'
zip_path = WORKING_DIR / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, p.relative_to(AGENT_DIR).as_posix())
print(f'Wrote {zip_path} ({zip_path.stat().st_size / 1024:.1f} KiB):')
for p in files:
    print('  ', p.relative_to(AGENT_DIR).as_posix())

## 3. Install the harness and validate
Skipped during the scoring re-run. The dev evaluation needs the **GPU L4 x4** accelerator, which can only be chosen in the notebook editor.

In [ ]:
import subprocess


def list_gpus():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=60)
    except (OSError, subprocess.TimeoutExpired):
        return []
    return [line.strip() for line in out.stdout.splitlines() if line.strip()] if out.returncode == 0 else []


if DEV and not GOLD_CHECK_ONLY:
    gpus = list_gpus()
    print(f'{len(gpus)} GPU(s): ' + '; '.join(gpus) if gpus else 'No GPU found.')
    assert gpus, (
        'No GPU attached. In the notebook editor open Settings > Accelerator, choose "GPU L4 x4", '
        'then Save Version > Save & Run All (runs started with `kaggle kernels push` cannot get L4 GPUs). '
        'To only build submission.zip, set RUN_DEV_EVAL = False.'
    )

In [ ]:
import glob
import importlib
import logging
import shutil
import subprocess
import sys

# Offline vLLM serving and LiteLLM routing (from the official starter notebook)
os.environ.update({
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_FLAX': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'VLLM_NO_USAGE_STATS': '1',
    'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
    'NCCL_DEBUG': 'WARN',
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'LITELLM_TELEMETRY': 'False',
    'LITELLM_LOG': 'WARNING',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OTEL_SDK_DISABLED': 'true',
    'OTEL_PYTHON_DISABLED': 'true',
})
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')
os.environ.setdefault('LOCAL_API_KEY', 'EMPTY')


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in ('LiteLLM', 'LiteLLM Router', 'LiteLLM Proxy', 'litellm', 'httpx', 'httpcore',
                 'openai', 'urllib3', 'opentelemetry', 'opentelemetry.context'):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


def install_wheelhouse() -> None:
    # Remove broken cutlass .pth hooks if present
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for pth in glob.glob(pattern):
            try:
                os.unlink(pth)
            except OSError:
                pass
    # Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
    tmp_whl = Path('/tmp/wheelhouse')
    tmp_whl.mkdir(parents=True, exist_ok=True)
    for w in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in w.name.lower():
            continue
        name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
        target = tmp_whl / name
        if not target.exists():
            try:
                os.symlink(w, target)
            except OSError:
                shutil.copyfile(w, target)
    wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
    print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
                   check=True)
    importlib.invalidate_caches()
    configure_logging()
    print('Wheelhouse installation complete.')


configure_logging()
if not IS_RERUN:
    install_wheelhouse()

In [ ]:
ALLOWED_MODEL_NAMES = {
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
}


def validate_submission():
    """Run the scorer's structure checks and compile the YAML with placeholder tools."""
    from adk_submission import ModelRegistry, compile_submission, validate_directory
    from adk_submission.discovery import discover_declared_models
    from swegemma.config import build_submission_limits

    limits, gen_constraints = build_submission_limits()
    validate_directory(AGENT_DIR, limits=limits)
    models = discover_declared_models(AGENT_DIR)
    assert len(models) == 1, f'Expected exactly 1 declared model, got {sorted(models)}'
    model = next(iter(models))
    assert model in ALLOWED_MODEL_NAMES, f'Model {model!r} is not allowed'

    tool_names = ['run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch',
                  'get_code_neighbors', 'search_similar_code', 'get_code_subgraph']

    def placeholder(name):
        def tool() -> str:
            return ''
        tool.__name__ = name
        tool.__doc__ = f'Placeholder for {name}.'
        return tool

    registry = ModelRegistry()
    registry.register(model, f'openai/{model}')
    agent = compile_submission(
        submission_dir=AGENT_DIR,
        tool_registry={n: placeholder(n) for n in tool_names},
        model_registry=registry,
        limits=limits,
        generation_constraints=gen_constraints,
    )
    print(f'Submission OK: {type(agent).__name__} {agent.name!r}, model={model}')
    return model


if not IS_RERUN:
    DECLARED_MODEL = validate_submission()

## 4. Dev evaluation on public tasks
Runs only when `RUN_DEV_EVAL = True` and this is not the scoring re-run. Expect ~10-20 min for vLLM startup plus up to `max_time_minutes` per task.

In [ ]:
import random
from collections import defaultdict


def pick_dev_tasks():
    from swegemma.models import load_tasks

    tasks = load_tasks(DATA_DIR / 'tasks.jsonl')
    if GOLD_CHECK_ONLY:
        return tasks
    if DEV_TASK_IDS:
        wanted = set(DEV_TASK_IDS)
        return [t for t in tasks if t.instance_id in wanted]
    by_repo = defaultdict(list)
    for t in tasks:
        by_repo[t.repo].append(t)
    rng = random.Random(0)
    picked = []
    for repo, n in DEV_TASKS_PER_REPO.items():
        picked += rng.sample(by_repo[repo], min(n, len(by_repo[repo])))
    return picked


if DEV:
    dev_tasks = pick_dev_tasks()
    print(f'{len(dev_tasks)} dev tasks:')
    for t in dev_tasks:
        print(f'  - {t.instance_id:<16} {t.repo}')

In [ ]:
import asyncio
import concurrent.futures
import time

import pandas as pd
import yaml

RESULTS_DIR = WORKING_DIR / 'results'


def run_sync(fn, *args, **kwargs):
    """Run an async function to completion, even inside the notebook's event loop."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(fn(*args, **kwargs))
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(fn(*args, **kwargs))).result()


def make_evaluator(models, adapter_manifest=None):
    """Evaluator configured like the scorer, with budgets read from eval_config.yaml."""
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import EvalConfig, build_submission_limits
    from swegemma.evaluate import Evaluator

    raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8')) or {}
    ev = raw.get('evaluation', raw)
    turns = ev.get('max_turns', ev.get('max_llm_calls'))
    limits, gen_constraints = build_submission_limits()
    return Evaluator(EvalConfig(
        tasks_path=DATA_DIR / 'tasks.jsonl',
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=RESULTS_DIR,
        submission_dir=AGENT_DIR,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 300)),
        verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
        max_time_minutes=float(ev.get('max_time_minutes', 60.0)),
        max_tool_calls=int(ev.get('max_tool_calls', 100)),
        max_turns=int(turns) if turns is not None else None,
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapter_manifest,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        graph_dir=DATA_DIR / 'graphs',
        embeddings_dir=DATA_DIR / 'embeddings',
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    ))


SNAPSHOT_TOOLS = ('run_command', 'edit_file', 'write_file')
# Diff of the working tree vs HEAD, built in a throwaway index so the agent's git state is untouched
SNAPSHOT_CMD = ('cd /workspace && export GIT_INDEX_FILE=/tmp/_anytime_idx && git read-tree HEAD '
                '&& git add -A && git diff --cached --binary HEAD')


def install_anytime_recorder():
    """Dev-only instrumentation: after every state-changing tool call, store the working-tree diff in
    results/anytime/<task>.jsonl. Replaying these snapshots through the grader gives the anytime curve
    P(resolved | agent stopped after call c)."""
    import functools
    import inspect
    from swegemma.context import SwegemmaContext

    if getattr(SwegemmaContext.create_tools, '_anytime', False):
        return
    original = SwegemmaContext.create_tools
    out_dir = RESULTS_DIR / 'anytime'
    out_dir.mkdir(parents=True, exist_ok=True)

    def record(ctx, tool_name):
        try:
            res = ctx.docker.exec(ctx.container_id, SNAPSHOT_CMD, timeout=60)  # same path as the tools
            diff = getattr(res, 'stdout', '') or ''
        except Exception as e:  # never let instrumentation break the agent
            diff = f'__snapshot_error__ {type(e).__name__}: {e}'
        last = getattr(ctx, '_anytime_last', None)
        if diff == last:
            return
        ctx._anytime_last = diff
        row = {'call': ctx.tool_calls_used, 'tool': tool_name,
               'elapsed_s': round(ctx.agent_elapsed_seconds or 0.0, 1), 'diff': diff}
        with open(out_dir / f'{ctx.task_id}.jsonl', 'a', encoding='utf-8') as f:
            f.write(json.dumps(row) + '\n')

    def wrap(ctx, name, fn):
        if inspect.iscoroutinefunction(fn):
            @functools.wraps(fn)
            async def wrapper(*args, **kwargs):
                result = await fn(*args, **kwargs)
                record(ctx, name)
                return result
        else:
            @functools.wraps(fn)
            def wrapper(*args, **kwargs):
                result = fn(*args, **kwargs)
                record(ctx, name)
                return result
        return wrapper

    def create_tools(self):
        tools = original(self)
        return {n: wrap(self, n, f) if n in SNAPSHOT_TOOLS else f for n, f in tools.items()}

    create_tools._anytime = True
    SwegemmaContext.create_tools = create_tools


def save_artifacts(subdir, instance_id, patch, test_output):
    """Keep the patch and the grading pytest output of a task for later diagnosis."""
    out = RESULTS_DIR / subdir
    out.mkdir(parents=True, exist_ok=True)
    (out / f'{instance_id}.patch').write_text(patch or '', encoding='utf-8')
    (out / f'{instance_id}.test.log').write_text(test_output or '', encoding='utf-8')

In [ ]:
def check_harness(tasks):
    """Grade each task's reference solution: every task should resolve if local grading works."""
    from adk_submission import ModelRegistry
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.harness.verification import verify_task

    evaluator = make_evaluator(ModelRegistry())
    rows = []
    for task in tasks:
        snapshot, base_snapshot, snapshot_patch = resolve_task_snapshot_paths(
            DATA_DIR / 'snapshots', task.instance_id, task.repo)
        start = time.time()
        r = run_sync(verify_task, evaluator.docker, evaluator.config, task, snapshot,
                     base_snapshot_path=base_snapshot, patch_path=snapshot_patch,
                     agent_patch=task.patch, start_time=time.perf_counter())
        save_artifacts('harness_check', task.instance_id, task.patch, r.test_output)
        rows.append({'instance_id': task.instance_id, 'gold_resolved': bool(r.resolved),
                     'test_exit_code': r.test_exit_code, 'seconds': round(time.time() - start, 1),
                     'error': r.error_message})
        print(f'  {task.instance_id:<16} gold_resolved={r.resolved} exit={r.test_exit_code} '
              f'{rows[-1]["seconds"]}s {r.error_message or ""}', flush=True)
    df = pd.DataFrame(rows)
    print(f'Reference solutions resolved: {int(df.gold_resolved.sum())}/{len(df)}')
    return df


if DEV and (RUN_HARNESS_CHECK or GOLD_CHECK_ONLY):
    harness_check = check_harness(dev_tasks)
    harness_check.to_csv(WORKING_DIR / 'gold_check.csv', index=False)

In [ ]:
import re


def start_vllm(model_name):
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    litellm.drop_params = True
    model_path = Path(f'/kaggle/input/models/google/gemma-4/other/{model_name}/2')
    assert model_path.exists(), f'Model weights not found at {model_path}. Attach the model as an input.'

    adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    adapter_map = getattr(adapters, 'adapters', None) or {}
    for name in adapter_map:
        assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]*', name), f'Unsafe adapter name {name!r}'
    ranks = [r for info in adapter_map.values() if (r := getattr(info, 'rank', None))]

    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    extra_args = {
        'gemma-4-12b-it-qat-w4a16-ct': (
            '--hf-overrides', '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
        ),
    }.get(model_name, ())

    cfg = VllmConfig(
        model=str(model_path),
        port=8000,
        host='127.0.0.1',
        tool_call_parser='gemma4',
        reasoning_parser='gemma4',
        max_model_len=32768,
        dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
        gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True,
        enable_lora=bool(adapter_map),
        max_loras=max(1, len(adapter_map)),
        max_lora_rank=max(ranks) if ranks else 128,
        scheduler_reserve_full_isl=False,
        tensor_parallel_size=tp_size,
        startup_timeout=60 * 20,
        extra_args=extra_args,
    )
    server = VllmServer(cfg, adapter_manifest=adapters)
    server.start()
    print(f'vLLM server started on {server.base_url} ({gpu_count} GPUs, tp={tp_size})')
    models = server.create_model_registry(
        aliases=[model_name], model_prefix='openai/', api_key=os.environ['LOCAL_API_KEY'],
    )
    return server, models, adapters


if DEV and not GOLD_CHECK_ONLY:
    if globals().get('server_instance') is not None:
        server_instance.stop()
    server_instance, model_registry, adapter_manifest = start_vllm(DECLARED_MODEL)

In [ ]:
def run_dev_eval(tasks):
    if RECORD_ANYTIME:
        install_anytime_recorder()
    evaluator = make_evaluator(model_registry, adapter_manifest)
    rows_path = WORKING_DIR / 'dev_results.jsonl'  # appended per task, so partial runs keep their results

    async def one(idx, task, sem):
        async with sem:
            print(f'[{idx}/{len(tasks)}] start {task.instance_id} ({task.repo})', flush=True)
            start = time.time()
            try:
                r = await evaluator.evaluate_task(task=task, task_index=idx, total_tasks=len(tasks))
                save_artifacts('agent', task.instance_id, r.agent_patch, r.test_output)
                row = {
                    'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
                    'tool_calls': r.tool_calls, 'patch_chars': len((r.agent_patch or '').strip()),
                    'test_exit_code': r.test_exit_code, 'seconds': round(r.duration_seconds, 1),
                    'status': r.status, 'error': r.error_message,
                }
            except Exception as e:  # keep going so one broken task does not end the run
                row = {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': False,
                       'seconds': round(time.time() - start, 1), 'error': f'{type(e).__name__}: {e}'}
            with open(rows_path, 'a', encoding='utf-8') as f:
                f.write(json.dumps(row, default=str) + '\n')
            print(f'    {task.instance_id} -> resolved={row["resolved"]} tool_calls={row.get("tool_calls")} '
                  f'patch_chars={row.get("patch_chars")} {row["seconds"]}s', flush=True)
            return row

    async def run_all():
        sem = asyncio.Semaphore(max(1, DEV_CONCURRENCY))
        return await asyncio.gather(*(one(i, t, sem) for i, t in enumerate(tasks, start=1)))

    rows = run_sync(run_all)
    df = pd.DataFrame(rows)
    df.to_csv(WORKING_DIR / 'dev_results.csv', index=False)
    print(f'\nResolved {int(df.resolved.sum())}/{len(df)} = {df.resolved.mean():.2%}')
    print(f'Per-task patches, test output and traces: {RESULTS_DIR}')
    return df


if DEV and not GOLD_CHECK_ONLY:
    dev_results = run_dev_eval(dev_tasks)
    display(dev_results)

In [ ]:
if globals().get('server_instance') is not None:
    server_instance.stop()
    server_instance = None
    print('vLLM server stopped.')